# Using the cortex ML inbuilt binary classificaton function

- Gradient boosting machine
- binary: AUC loss functino


#### Preparing training data

Doing a 80/20 split on the training_data table to have two seperate datasets for training and testing. 
Doing this not randomly, but by taking the first 80% rows for training and the last 20% for test. 


Then selecting the appropriate columns. 

In [ ]:
create or replace view temp_table as
select *, ROW_NUMBER() OVER (ORDER BY RANDOM()) AS row_num
from training_data;

-- Create the 80% sample view
create or replace view sample_80 as
select chain_cat_1, chain_cat_2, chain_cat_3, offer_value_1, offer_value_2, offer_value_3, offer_value_4, offer_value_5, offer_value_6, previous_purchase_category_int, previous_purchase_int, repeater_int
from temp_table
where row_num <= (SELECT COUNT(*) * 0.8 FROM temp_table);

select * from sample_80 limit 2;

-- Create the 20% sample view
create or replace view sample_20 as
select chain_cat_1, chain_cat_2, chain_cat_3, offer_value_1, offer_value_2, offer_value_3, offer_value_4, offer_value_5, offer_value_6, previous_purchase_category_int, previous_purchase_int, repeater_int
from temp_table
where row_num >= (SELECT COUNT(*) * 0.8 FROM temp_table);

select * from sample_20 limit 2;

In [ ]:
select count(*) from sample_20;

Checking the count of sample 80 and 20. 

In [ ]:
select count(*) from sample_80;

In [ ]:
select count(*) from sample_20;

### Creating model

Creating and training the model on sample_20 with the label=repeater_int

In [ ]:
create or replace snowflake.ml.classification model_binary(
    input_data => system$reference('view', 'sample_80'),
    target_colname => 'repeater_int'
);

### Predictions and metrics

Using the PREDICT function to make predictions on the test set and display the results with its corresponding input features. 

In [ ]:
select *, model_binary!PREDICT(
    INPUT_DATA => {*})
    as predictions from sample_20;

Various evaluation metrics. Copied into markdown as well such that it is saved. 

In [ ]:
CALL model_binary!SHOW_EVALUATION_METRICS();

In [ ]:
CALL model_binary!SHOW_GLOBAL_EVALUATION_METRICS();

In [ ]:
CALL model_binary!SHOW_THRESHOLD_METRICS();

In [ ]:
CALL model_binary!SHOW_CONFUSION_MATRIX();

In [ ]:
CALL model_binary!SHOW_FEATURE_IMPORTANCE();